# Imports

In [ ]:
import json
from contextlib import ExitStack
from dataclasses import replace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from uavloc.ardupilot.flight import FeedConfig, FlightSession
from uavloc.ardupilot.link import (HOME_WARNING_M, connect, download_mission, get_param, home_offset_m, set_params,
                                   upload_mission, vehicle_home)
from uavloc.ardupilot.mission import (MissionPlan, mission_differences, mission_items, read_waypoints,
                                      write_waypoints)
from uavloc.ardupilot.params import read_param_file, vehicle_params, write_param_file
from uavloc.ardupilot.scenarios import check_gps_input, enable_auto_takeoff, fly_mission, run_coast_test
from uavloc.ardupilot.scripts import lua_script_path
from uavloc.ardupilot.sitl import SitlConfig, default_ardupilot_dir, start_sitl
from uavloc.builds import build_dir
from uavloc.data.tiles import ESRI_WORLD_IMAGERY
from uavloc.features.onnx_segmenter import OnnxSegmenter
from uavloc.geo import LatLon, east_north_m_array, haversine_m
from uavloc.planning.config import MissionInputs, PlannerConfig
from uavloc.planning.localizability import SelfSimilarity
from uavloc.planning.planner import plan
from uavloc.track import REPO_ROOT

# Params

In [ ]:
SLUG = "ardupilot-mission"
START = (49.905, 23.885)        # west of Birky
END = (49.915, 23.925)          # east of Birky
ALTITUDE_M = 40.0
SPEED_MPS = 5.0
HOVER_S = 0.0
MAX_FLIGHT_TIME_S = 1200.0
SATELLITE_ONNX = "pocs/lraspp-pi5/models/lraspp-satellite.onnx"
TILE_CACHE = "data/derived/tile-cache"
PLANNER_CACHE = "data/derived/planner-cache"
HOME_ALT_M = 300.0              # AMSL of the simulated home
HEADING_DEG = 90.0
SPEEDUP = 8.0
FLIGHT_TIMEOUT_S = 4000.0       # simulated seconds
COAST_OBSERVE_S = 60.0          # simulated seconds watched after the last fix when the EKF failsafe is off
FEED_NOISE_STD_M = 0.0

# Data

In [ ]:
build = build_dir(SLUG, create=True)
figures, exports, runs = build / "figures", build / "exports", build / "runs"
inputs = MissionInputs(LatLon(*START), LatLon(*END), ALTITUDE_M, SPEED_MPS, MAX_FLIGHT_TIME_S, HOVER_S)
config = PlannerConfig()
ardupilot_dir = default_ardupilot_dir()
print(f"start-end {haversine_m(inputs.start, inputs.end):.0f} m; ArduPilot at {ardupilot_dir}")

# Model

In [ ]:
segmenter = OnnxSegmenter(REPO_ROOT / SATELLITE_ONNX)
localizability = SelfSimilarity(config.matcher, config.match_gsd_m, config.blind_s_max_m, config.blind_c_min)

# Route

In [ ]:
result = plan(inputs, config, ESRI_WORLD_IMAGERY, segmenter, localizability,
              REPO_ROOT / TILE_CACHE, REPO_ROOT / PLANNER_CACHE)
route = min(result.routes, key=lambda candidate: candidate.blind_share)
print(f"{len(result.routes)} routes; chosen weight {route.weight:g}, {route.length_m:.0f} m, "
      f"blind share {route.blind_share:.3f}, {len(route.waypoints)} waypoints")

In [ ]:
mission_plan = MissionPlan.from_route(route, inputs)
items = mission_items(mission_plan)
print(len(items), "mission items")

# Export

In [ ]:
waypoints_path, params_path = exports / "mission.waypoints", exports / "uavloc.param"
write_waypoints(waypoints_path, items)
params = vehicle_params()
write_param_file(params_path, params)
(exports / lua_script_path().name).write_text(lua_script_path().read_text())
print(sorted(path.name for path in exports.iterdir()))

In [ ]:
assert mission_differences(items, read_waypoints(waypoints_path)) == []
assert read_param_file(params_path) == params

# Shadow flight

In [ ]:
sitl_config = SitlConfig(ardupilot_dir, inputs.start, HOME_ALT_M, HEADING_DEG, SPEEDUP, params,
                         work_dir=runs / "sitl-flight")
stack = ExitStack()
url = stack.enter_context(start_sitl(sitl_config))
conn = connect(url, timeout_s=30.0)
print(url, "system", conn.target_system)

In [ ]:
session = FlightSession(conn, FeedConfig(noise_std_m=FEED_NOISE_STD_M))
session.request_streams()

In [ ]:
set_params(conn, params)
readback = pd.DataFrame({"set": params, "vehicle": {name: get_param(conn, name) for name in params}})
readback

In [ ]:
enable_auto_takeoff(session)
home = vehicle_home(conn, timeout_s=120.0)
offset_m = home_offset_m(home, inputs.start)
print(f"vehicle home {home}; {offset_m:.2f} m from the planned start (warn above {HOME_WARNING_M:g} m)")

In [ ]:
upload_mission(conn, items)
downloaded = download_mission(conn)
differences = mission_differences(items, downloaded)
print(len(downloaded), "items on the vehicle; differences:", differences or "none")

In [ ]:
gps_check = check_gps_input(session)
print(gps_check, "passed" if gps_check.passed else "FAILED")

# Run

In [ ]:
flight = fly_mission(session, inputs.start, inputs.end, FLIGHT_TIMEOUT_S)
print(f"{flight.duration_sim_s:.0f} simulated s, max height {flight.max_height_m:.1f} m, "
      f"closest to end point {flight.closest_to_end_m:.1f} m, landed {flight.landed_offset_m(inputs.start):.2f} m from start")

In [ ]:
stack.close()

# Coasting

In [ ]:
COAST_VARIANTS = {
    "vision, failsafe on": dict(use_vision=True, ekf_action=1, observe_s=COAST_OBSERVE_S),
    "vision, failsafe off": dict(use_vision=True, ekf_action=0, observe_s=COAST_OBSERVE_S),
    "real GPS lost, failsafe on": dict(use_vision=False, ekf_action=1, observe_s=COAST_OBSERVE_S),
}

In [ ]:
def coast(name: str):
    work_dir = runs / f"sitl-coast-{name.replace(' ', '-').replace(',', '')}"
    return run_coast_test(replace(sitl_config, work_dir=work_dir), items, **COAST_VARIANTS[name])

In [ ]:
coasts = {name: coast(name) for name in COAST_VARIANTS}

In [ ]:
def coast_row(name: str, outcome) -> dict:
    drift = [error for seconds, error in outcome.error_m if seconds <= 10.0]
    return {"variant": name, "position_lost_s": outcome.position_lost_s, "failsafe_s": outcome.failsafe_s,
            "mode_after": outcome.mode_after_failsafe, "error_max_first_10s_m": max(drift)}

In [ ]:
coast_table = pd.DataFrame([coast_row(name, outcome) for name, outcome in coasts.items()])
coast_table

# Plots

In [ ]:
def to_enu(points) -> np.ndarray:
    east_m, north_m = east_north_m_array(inputs.start, [p.lat for p in points], [p.lon for p in points])
    return np.column_stack([east_m, north_m])

In [ ]:
def plot_track(ax) -> None:
    planned = to_enu([inputs.start, *route.waypoints[1:], *route.waypoints[-2::-1]])
    truth = to_enu([p.truth for p in flight.track])
    estimate = to_enu([p.estimate for p in flight.track])
    ax.plot(*planned.T, "k--", lw=1.5, label="planned out and back")
    ax.plot(*truth.T, lw=1.2, label="SITL truth")
    ax.plot(*estimate.T, lw=0.8, label="EKF estimate")
    ax.plot(*to_enu([inputs.start]).T, "go", label="start / landing")
    ax.plot(*to_enu([inputs.end]).T, "r^", label="end point")
    ax.set_aspect("equal")
    ax.set_xlabel("east of start, m")
    ax.set_ylabel("north of start, m")
    ax.legend(fontsize=8)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6), constrained_layout=True)
plot_track(ax)
ax.set_title(f"AUTO mission on SITL GPS, landed {flight.landed_offset_m(inputs.start):.2f} m from start")
fig.savefig(figures / "track-over-route.png", dpi=130)

In [ ]:
def plot_height_and_error(axes) -> None:
    seconds = np.array([p.sim_time_s for p in flight.track]) - flight.track[0].sim_time_s
    axes[0].plot(seconds, [p.height_m for p in flight.track])
    axes[0].set_ylabel("height above takeoff, m")
    axes[1].plot(seconds, [haversine_m(p.truth, p.estimate) for p in flight.track])
    axes[1].set_ylabel("EKF error vs truth, m")
    axes[1].set_xlabel("simulated seconds since arming")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True, constrained_layout=True)
plot_height_and_error(axes)
fig.savefig(figures / "height-and-ekf-error.png", dpi=130)

In [ ]:
def plot_coast(ax) -> None:
    for name, outcome in coasts.items():
        seconds, errors = zip(*outcome.error_m)
        ax.plot(seconds, errors, label=name)
        if outcome.failsafe_s is not None:
            ax.axvline(outcome.failsafe_s, ls=":", color="grey")
    ax.set_yscale("log")
    ax.set_xlabel("seconds after the last fix")
    ax.set_ylabel("EKF position error vs truth, m")
    ax.legend()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5), constrained_layout=True)
plot_coast(ax)
ax.set_title("IMU-only coasting at 5 m/s (dotted: EKF failsafe to Land)")
fig.savefig(figures / "imu-coasting.png", dpi=130)

# Results

In [ ]:
summary = {
    "route_weight": route.weight, "route_length_m": route.length_m, "route_blind_share": route.blind_share,
    "waypoints": len(route.waypoints), "mission_items": len(items), "mission_differences": differences,
    "home_offset_m": offset_m, "gps_input_check": {"fixes_seen": gps_check.fixes_seen,
                                                    "max_error_m": gps_check.max_error_m},
    "flight_sim_s": flight.duration_sim_s, "max_height_m": flight.max_height_m,
    "closest_to_end_m": flight.closest_to_end_m, "landed_offset_m": flight.landed_offset_m(inputs.start),
    "coast": json.loads(coast_table.to_json(orient="records")),
}
(build / "results.json").write_text(json.dumps(summary, indent=2))
summary